# Filter Tracks by ROI

Batch-filters TrackMate tracks CSVs by ROI: for every spots/tracks CSV pair in a folder
(matched by their shared leading numeric date prefix), keeps only the tracks whose first
frame starts inside that folder's ROI mask -- a binary TIFF the same pixel size as the movie.

Used as a preprocessing step before the B cell/macrophage clustering pipeline
(`BCellClustering.ipynb` / `MacrophageClustering.ipynb`).

## Configuration
Edit `FOLDER_PATH` below (and `ROI_FILE`/`SUFFIX` if needed), then run top to bottom.
`ROI_FILE = None` auto-detects the mask TIFF in `FOLDER_PATH` -- preferring a filename
containing "mask" if the folder has more than one TIFF, since a replicate folder can also
hold raw per-channel movie TIFFs (e.g. `Phagy Count Sorted Data/{size}/R{n}/`, which has both
`BCellM0_..._mask_whole.tif` and separate per-channel movie TIFFs side by side).

In [1]:
# ─── IMPORTS ───────────────────────────────────────────────────────────────
import re
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

# ─── CONFIG ────────────────────────────────────────────────────────────────
BASE_DIR = Path.cwd()

# Example: one replicate folder from `Phagy Count Sorted Data` (spots + tracks CSVs and
# the ROI mask TIFF all live together per size/replicate).
FOLDER_PATH = BASE_DIR / "Phagy Count Sorted Data" / "90 µm" / "R1"
ROI_FILE = None       # filename within FOLDER_PATH, or None to auto-detect
SUFFIX = "_filtered_tracks"

# Filtered output is written here, NOT alongside the source files -- keeps this
# notebook's runs from writing into `Phagy Count Sorted Data`.
OUTPUT_DIR = BASE_DIR / "Testing" / "FilterByROI_output"


## Filtering function

In [2]:
# ─── FILTER TRACKS BY ROI ────────────────────────────────────────────────────
def points_inside_mask(x, y, mask):
    """Boolean array: True where (x, y), rounded to the nearest pixel, falls
    inside the bounds of `mask` and that pixel is truthy."""
    col_idx = np.round(x).astype(int)
    row_idx = np.round(y).astype(int)
    in_bounds = (row_idx >= 0) & (row_idx < mask.shape[0]) & (col_idx >= 0) & (col_idx < mask.shape[1])
    inside = np.zeros(len(x), dtype=bool)
    inside[in_bounds] = mask[row_idx[in_bounds], col_idx[in_bounds]]
    return inside

def find_roi_mask(folder_path: Path, roi_file: str = None) -> Path:
    """Resolve the ROI mask TIFF for `folder_path`. If `roi_file` is given, use it
    directly; otherwise auto-detect, preferring a filename containing "mask" since a
    replicate folder can also hold raw per-channel movie TIFFs alongside the mask."""
    if roi_file:
        mask_path = folder_path / roi_file
        if not mask_path.is_file():
            raise FileNotFoundError(f"Specified ROI file not found: {mask_path}")
        return mask_path

    tiff_files = sorted(folder_path.glob("*.tif")) + sorted(folder_path.glob("*.tiff"))
    if not tiff_files:
        raise FileNotFoundError(f"No TIFF file found in {folder_path} for ROI mask. Provide one via ROI_FILE.")

    mask_candidates = [f for f in tiff_files if "mask" in f.name.lower()]
    if mask_candidates:
        if len(mask_candidates) > 1:
            print(f"[WARN] Multiple mask-named TIFFs found, using first: {mask_candidates[0].name}")
        return mask_candidates[0]

    if len(tiff_files) > 1:
        print(f"[WARN] Multiple TIFFs found, none named \'mask\'; using first: {tiff_files[0].name}. "
              f"Pass ROI_FILE explicitly to avoid ambiguity.")
    return tiff_files[0]

def filter_tracks_by_roi(folder_path: Path, output_dir: Path, roi_file: str = None, suffix: str = "_filtered_tracks"):
    """For every matching spots/tracks CSV pair in `folder_path` (matched by shared
    leading numeric date prefix), keep only the tracks starting inside the folder's ROI
    mask in the first frame. Saves one filtered tracks CSV per pair into `output_dir`
    (created if needed, NOT alongside the source files); returns their paths."""
    mask_path = find_roi_mask(folder_path, roi_file)
    mask = tifffile.imread(mask_path).astype(bool)
    print(f"Loaded ROI mask: {mask_path.name} (shape {mask.shape})")

    spots_files = list(folder_path.glob("*spots*.csv"))
    tracks_files = list(folder_path.glob("*tracks*.csv"))

    def get_prefix(path: Path):
        m = re.match(r"^(\d+)", path.name)
        return m.group(1) if m else None

    spots_map = {get_prefix(f): f for f in spots_files if get_prefix(f)}
    tracks_map = {get_prefix(f): f for f in tracks_files if get_prefix(f)}

    output_dir.mkdir(parents=True, exist_ok=True)
    out_paths = []
    for prefix, tfile in tracks_map.items():
        sfile = spots_map.get(prefix)
        if not sfile:
            print(f"Skipping {tfile.name}: no matching spots CSV.")
            continue

        print(f"\nProcessing files for prefix {prefix}:")
        print(f"  Spots:  {sfile.name}")
        print(f"  Tracks: {tfile.name}")

        spots = pd.read_csv(sfile, low_memory=False)
        tracks = pd.read_csv(tfile, low_memory=False)

        for col in ["FRAME", "POSITION_X", "POSITION_Y", "TRACK_ID"]:
            if col not in spots.columns:
                raise KeyError(f"Column '{col}' not found in {sfile.name}")
            spots[col] = pd.to_numeric(spots[col], errors="coerce")
        spots = spots.dropna(subset=["FRAME", "POSITION_X", "POSITION_Y", "TRACK_ID"])
        spots["FRAME"] = spots["FRAME"].astype(int)
        spots["TRACK_ID"] = spots["TRACK_ID"].astype(int)

        if "TRACK_ID" not in tracks.columns:
            raise KeyError(f"Column 'TRACK_ID' not found in {tfile.name}")
        tracks["TRACK_ID"] = pd.to_numeric(tracks["TRACK_ID"], errors="coerce")
        tracks = tracks.dropna(subset=["TRACK_ID"])
        tracks["TRACK_ID"] = tracks["TRACK_ID"].astype(int)

        first_frame = spots["FRAME"].min()
        first_spots = spots[spots["FRAME"] == first_frame]

        keep = points_inside_mask(first_spots["POSITION_X"].to_numpy(), first_spots["POSITION_Y"].to_numpy(), mask)
        inside_set = set(first_spots.loc[keep, "TRACK_ID"])
        print(f"Found {len(inside_set)} tracks beginning inside ROI.")

        filtered = tracks[tracks["TRACK_ID"].isin(inside_set)]
        out_path = output_dir / f"{tfile.stem}{suffix}{tfile.suffix}"
        filtered.to_csv(out_path, index=False)
        print(f"Saved filtered tracks: {out_path}")
        out_paths.append(out_path)

    return out_paths


## Run

In [3]:
filter_tracks_by_roi(FOLDER_PATH, OUTPUT_DIR, ROI_FILE, SUFFIX)


Loaded ROI mask: BCellM0_90um_R1_mask_whole.tif (shape (1200, 1600))

Processing files for prefix 20240911:
  Spots:  20240911_90um_Y_R1_spots_unfiltered.csv
  Tracks: 20240911_90um_Y_R1_tracks_unfiltered.csv


Found 961 tracks beginning inside ROI.
Saved filtered tracks: /Users/brianmah/Claude/MCP Clustering Workspace/Testing/FilterByROI_output/20240911_90um_Y_R1_tracks_unfiltered_filtered_tracks.csv


[PosixPath('/Users/brianmah/Claude/MCP Clustering Workspace/Testing/FilterByROI_output/20240911_90um_Y_R1_tracks_unfiltered_filtered_tracks.csv')]